In [57]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
import easygui
from collections import defaultdict
import plotly.express as px
import numpy as np

In [58]:
files = easygui.fileopenbox(filetypes="*.csv", multiple=True)

In [59]:
def maxSubplot():
    files = easygui.fileopenbox(filetypes="*.txt", multiple=True)
    # Ось X берём из первого файла и приводим к datetime
    x_raw = pd.read_csv(files[0])['Дата']
    xaxis = pd.to_datetime(x_raw, errors='coerce')
    fig = go.Figure()
    for f in files:
        df = pd.read_csv(f)
        y = df['max']
        try:
            series_name = Path(f).stem.split('-')[1]
        except Exception:
            series_name = Path(f).stem
        fig.add_trace(go.Scatter(x=xaxis, y=y, name=series_name,line_shape='hv', mode='lines+markers'))
    fig.update_layout(
        title="График максимальных суточных температур",
        xaxis=dict(title="Дата", rangeslider=dict(visible=False)),
        yaxis=dict(title="Температура", fixedrange=False),
        dragmode="drawline",                 # режим рисования линий по умолчанию
        newshape=dict(line_color="red"),     # цвет рисуемых линий
    )
    config = dict(modeBarButtonsToAdd=["drawline", "drawopenpath", "eraseshape"])
    html = pio.to_html(fig, include_plotlyjs='cdn', full_html=True, config=config)
    panel_and_js = r"""
<style>
  #axis-toggle {
    position: fixed; right: 16px; bottom: 16px; z-index: 10000;
    background: #2b6cb0; color: #fff; border: none; border-radius: 999px;
    padding: 10px 14px; font: 13px/1 sans-serif; cursor: pointer;
    box-shadow: 0 2px 8px rgba(0,0,0,.2);
  }
  #axis-panel {
    position: fixed; right: 16px; bottom: 64px; z-index: 10000;
    width: 320px; max-width: calc(100vw - 32px);
    background: rgba(255,255,255,.98); border: 1px solid #ddd; border-radius: 12px;
    padding: 12px; font: 12px/1.3 sans-serif; box-shadow: 0 6px 16px rgba(0,0,0,.2);
    transform: translateY(12px); opacity: 0; pointer-events: none;
    transition: transform .18s ease, opacity .18s ease;
  }
  #axis-panel.open {
    transform: translateY(0); opacity: 1; pointer-events: auto;
  }
  #axis-panel .grid {
    display: grid; grid-template-columns: auto auto; gap: 6px; align-items: center;
  }
  #axis-panel input { width: 150px; }
  #axis-panel .row { margin-top: 8px; display: flex; gap: 8px; }
  #axis-panel .hint { margin-top: 6px; color: #777; }
  #axis-panel .title { margin-bottom: 6px; font-weight: 600; }
</style>

<button id="axis-toggle" title="Показать/скрыть панель диапазонов">Диапазоны</button>

<div id="axis-panel" aria-hidden="true">
  <div class="title">Диапазоны осей</div>
  <div class="grid">
    <label>X min</label><input id="x_min" type="text" placeholder="auto">
    <label>X max</label><input id="x_max" type="text" placeholder="auto">
    <label>Y min</label><input id="y_min" type="number" step="any" placeholder="auto">
    <label>Y max</label><input id="y_max" type="number" step="any" placeholder="auto">
  </div>
  <div class="row">
    <button id="apply_btn">Применить</button>
    <button id="reset_btn">Сброс</button>
  </div>
  <div class="hint">Дата: 2021-06-01 или 01.06.2021 · Горячая клавиша: <b>Alt+D</b></div>
</div>

<script>
(function(){
  const panel = document.getElementById('axis-panel');
  const btn   = document.getElementById('axis-toggle');

  function toggle(open){
    const willOpen = (open !== undefined) ? open : !panel.classList.contains('open');
    panel.classList.toggle('open', willOpen);
    panel.setAttribute('aria-hidden', String(!willOpen));
  }

  btn.addEventListener('click', () => toggle());
  // Горячая клавиша Alt+D — открыть/закрыть (с учётом русской раскладки)
  window.addEventListener('keydown', (e) => {
    if (e.altKey && (e.key === 'd' || e.key === 'в')) {
      e.preventDefault(); toggle();
    }
  });

  function getPlotDiv(){ return document.querySelector('.js-plotly-plot'); }
  function parseX(v){
    if(!v) return null; v = v.trim(); if(!v) return null;
    const isDate = /^\d{4}-\d{2}-\d{2}/.test(v) || /^\d{2}\.\d{2}\.\d{4}/.test(v);
    if(isDate) return v;                     // даты Plotly распарсит сам
    const n = Number(v);
    return Number.isFinite(n) ? n : v;       // число или категория
  }

  document.getElementById('apply_btn').onclick = function(){
    const gd = getPlotDiv(); if(!gd) return;

    const xMin = parseX(document.getElementById('x_min').value);
    const xMax = parseX(document.getElementById('x_max').value);
    const yMinStr = document.getElementById('y_min').value.trim();
    const yMaxStr = document.getElementById('y_max').value.trim();

    const yMin = yMinStr === '' ? null : Number(yMinStr);
    const yMax = yMaxStr === '' ? null : Number(yMaxStr);

    const relayout = {};
    if (xMin != null || xMax != null) relayout['xaxis.range'] = [xMin, xMax];
    if (yMin != null || yMax != null) relayout['yaxis.range'] = [yMin, yMax];

    if (Object.keys(relayout).length) {
      Plotly.relayout(gd, relayout);
    }
  };

  document.getElementById('reset_btn').onclick = function(){
    const gd = getPlotDiv(); if(!gd) return;
    Plotly.relayout(gd, {'xaxis.autorange': true, 'yaxis.autorange': true});
  };
})();
</script>
"""
    html = html.replace("</body>", panel_and_js + "\n</body>")
    out_name = f"{f.split('-')[0]}max.html"
    with open(out_name, "w", encoding="utf-8") as f:
        f.write(html)

In [60]:
def minSubplot():
    x_raw = pd.read_csv(files[0])['Дата']
    xaxis = pd.to_datetime(x_raw, errors='coerce')
    fig = go.Figure()
    for f in files:
        df = pd.read_csv(f)
        y = df['max']
        try:
            series_name = Path(f).stem.split('-')[1]
        except Exception:
            series_name = Path(f).stem

        fig.add_trace(go.Scatter(
            x=xaxis, y=y, name=series_name,
            line_shape='hv', mode='lines+markers'
        ))
    fig.update_layout(
        title="График максимальных суточных температур",
        xaxis=dict(title="Дата", rangeslider=dict(visible=False)),
        yaxis=dict(title="Температура", fixedrange=False),
        dragmode="drawline",                 # режим рисования линий по умолчанию
        newshape=dict(line_color="red"),     # цвет рисуемых линий
    )
    config = dict(modeBarButtonsToAdd=["drawline", "drawopenpath", "eraseshape"])
    html = pio.to_html(fig, include_plotlyjs='cdn', full_html=True, config=config)
    panel_and_js = r"""
<style>
  #axis-toggle {
    position: fixed; right: 16px; bottom: 16px; z-index: 10000;
    background: #2b6cb0; color: #fff; border: none; border-radius: 999px;
    padding: 10px 14px; font: 13px/1 sans-serif; cursor: pointer;
    box-shadow: 0 2px 8px rgba(0,0,0,.2);
  }
  #axis-panel {
    position: fixed; right: 16px; bottom: 64px; z-index: 10000;
    width: 320px; max-width: calc(100vw - 32px);
    background: rgba(255,255,255,.98); border: 1px solid #ddd; border-radius: 12px;
    padding: 12px; font: 12px/1.3 sans-serif; box-shadow: 0 6px 16px rgba(0,0,0,.2);
    transform: translateY(12px); opacity: 0; pointer-events: none;
    transition: transform .18s ease, opacity .18s ease;
  }
  #axis-panel.open {
    transform: translateY(0); opacity: 1; pointer-events: auto;
  }
  #axis-panel .grid {
    display: grid; grid-template-columns: auto auto; gap: 6px; align-items: center;
  }
  #axis-panel input { width: 150px; }
  #axis-panel .row { margin-top: 8px; display: flex; gap: 8px; }
  #axis-panel .hint { margin-top: 6px; color: #777; }
  #axis-panel .title { margin-bottom: 6px; font-weight: 600; }
</style>

<button id="axis-toggle" title="Показать/скрыть панель диапазонов">Диапазоны</button>

<div id="axis-panel" aria-hidden="true">
  <div class="title">Диапазоны осей</div>
  <div class="grid">
    <label>X min</label><input id="x_min" type="text" placeholder="auto">
    <label>X max</label><input id="x_max" type="text" placeholder="auto">
    <label>Y min</label><input id="y_min" type="number" step="any" placeholder="auto">
    <label>Y max</label><input id="y_max" type="number" step="any" placeholder="auto">
  </div>
  <div class="row">
    <button id="apply_btn">Применить</button>
    <button id="reset_btn">Сброс</button>
  </div>
  <div class="hint">Дата: 2021-06-01 или 01.06.2021 · Горячая клавиша: <b>Alt+D</b></div>
</div>

<script>
(function(){
  const panel = document.getElementById('axis-panel');
  const btn   = document.getElementById('axis-toggle');

  function toggle(open){
    const willOpen = (open !== undefined) ? open : !panel.classList.contains('open');
    panel.classList.toggle('open', willOpen);
    panel.setAttribute('aria-hidden', String(!willOpen));
  }

  btn.addEventListener('click', () => toggle());
  // Горячая клавиша Alt+D — открыть/закрыть (с учётом русской раскладки)
  window.addEventListener('keydown', (e) => {
    if (e.altKey && (e.key === 'd' || e.key === 'в')) {
      e.preventDefault(); toggle();
    }
  });

  function getPlotDiv(){ return document.querySelector('.js-plotly-plot'); }
  function parseX(v){
    if(!v) return null; v = v.trim(); if(!v) return null;
    const isDate = /^\d{4}-\d{2}-\d{2}/.test(v) || /^\d{2}\.\d{2}\.\d{4}/.test(v);
    if(isDate) return v;                     // даты Plotly распарсит сам
    const n = Number(v);
    return Number.isFinite(n) ? n : v;       // число или категория
  }

  document.getElementById('apply_btn').onclick = function(){
    const gd = getPlotDiv(); if(!gd) return;

    const xMin = parseX(document.getElementById('x_min').value);
    const xMax = parseX(document.getElementById('x_max').value);
    const yMinStr = document.getElementById('y_min').value.trim();
    const yMaxStr = document.getElementById('y_max').value.trim();

    const yMin = yMinStr === '' ? null : Number(yMinStr);
    const yMax = yMaxStr === '' ? null : Number(yMaxStr);

    const relayout = {};
    if (xMin != null || xMax != null) relayout['xaxis.range'] = [xMin, xMax];
    if (yMin != null || yMax != null) relayout['yaxis.range'] = [yMin, yMax];

    if (Object.keys(relayout).length) {
      Plotly.relayout(gd, relayout);
    }
  };

  document.getElementById('reset_btn').onclick = function(){
    const gd = getPlotDiv(); if(!gd) return;
    Plotly.relayout(gd, {'xaxis.autorange': true, 'yaxis.autorange': true});
  };
})();
</script>
"""
    html = html.replace("</body>", panel_and_js + "\n</body>")
    out_name = f"{f.split('-')[0]}minx.html"
    with open(out_name, "w", encoding="utf-8") as f:
        f.write(html)

In [61]:
def seasonPlot():
    #files = easygui.fileopenbox(filetypes="*.csv", multiple=True)
    dictionary = defaultdict(list)
    for name in files:
        csv = pd.read_csv(name)
        csv['year'] = [x.strftime('%Y') for x in csv['Дата'].astype('datetime64[ns]')]
        csv['month'] = [x.strftime('%B') for x in csv['Дата'].astype('datetime64[ns]')]
        monthCategory = pd.CategoricalDtype(
            categories=["January", "February", "March", "April", "May", "June", 
                        "July", "August", "September", "October", "November", "December"],ordered=True)
        csv = csv.groupby(by=['year', "month"])[['max']].max().reset_index()
        csv= csv[(csv['month'] == 'June')|(csv['month'] == 'December')].drop(1).reset_index(drop=True)
        csv['month'] = csv['month'].astype(monthCategory)
        csv['season'] = ['winter' if 'December' in x else 'summer' for x in csv['month']]
        csv = csv.sort_values(['year', 'month']).reset_index(drop=True)
        csv['year'] = [x + '-' + y for x, y in zip(csv['year'], csv['month'])]
        csv.rename(columns = {'year':'date'}, inplace = True)
        csv.drop(columns = ['month'], inplace = True)
        csv['date'] = [x.strftime('%Y-%B') for x in csv['date'].astype('datetime64[ns]')]
        dictionary[name.split('-')[1]].append(csv)
        
    ###  each parameter in separated plot
    for name in dictionary.keys():
        year = []
        deltaS = []
        deltaW = []
        summer = dictionary[name][0][dictionary[name][0]['season'].isin(['summer'])].reset_index().drop(['index', 'season'], axis = 1)
        for x in range(len(summer['date'])):
            if x < len(summer['date']) - 1:
                year.append(summer['date'].iloc[x+1].split('-')[0]+'-'+ summer['date'].iloc[x].split('-')[0])
                deltaS.append(summer['max'].iloc[x+1] - summer['max'].iloc[x])
        Increment = pd.DataFrame()
        Increment['year'] = year
        winter = dictionary[name][0][dictionary[name][0]['season'].isin(['winter'])].reset_index().drop(['index', 'season'], axis = 1)
        for x in range(len(winter['date'])):
            if x < len(winter['date']) - 1:
                deltaW.append(winter['max'].iloc[x+1] - winter['max'].iloc[x])
        Increment['deltaS'] = deltaS
        Increment['deltaW'] = deltaW + [np.nan]
        #increment[name].append(Increment)
        fig = px.bar(Increment, x = 'year', y = ['deltaW','deltaS'], barmode='group', labels={"variable": "Season"})
        fig.update_layout(
            title="Приращение Температуры от года к году по сезонам",
            xaxis=dict(title="Дата", rangeslider=dict(visible=False)),
            yaxis=dict(title= "Δ °C", fixedrange=False),
            )
        fig.write_html(f"{files[0].split('-')[0]}{name}_season_increment.html")    
    ### selected parameters in one plot 
    fig = go.Figure()        
    for f in dictionary:
        df = dictionary[f][0]
        xaxis = df['date']
        y = df['max']
        fig.add_trace(go.Bar(x=xaxis, y=y, name=f)) 
    fig.update_layout(
        title="График максимальных сезонных температур",
        xaxis=dict(title="Дата", rangeslider=dict(visible=False)),
        yaxis=dict(title="Температура", fixedrange=False),
    )
    fig.write_html(f"{files[0].split('-')[0]}Season.html")   

In [63]:
#maxSubplot()
#minSubplot()
maxSeasonSubplot()
seasonPlot()